# Cohorte salud mental

El siguiente codigo construira toda la cohorte de salud mental, la intención es determinar cada una de las fuentes e ir construyendo la logica que llevara.

## Fuentes

- AVICENA
- SIVIGILA
- COHORTE ANTERIOR

## Salidas

- Cohorte mes en curso
- Egresos
- Egresos especifico
- Estado de enfermedad
- Medicamentos
- Población EPS
- Total atenciones

In [1]:
# librerias
import json
import os
import logging
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta


In [2]:
# Rutas fuentes
ruta_usuarios_finales = r'G:\.shortcut-targets-by-id\1oRHIu53FAkLguDxvW2JyAR5NSjEoMcXS\2026'
ruta_utilizaciones=r'G:\.shortcut-targets-by-id\1uyoVEDDA_-tskYM08AbRYIVn64VmCNQm\Bases\01.Utilizaciones\2026\UTILIZACIONES_202601.txt'
ruta_autorizaciones=r'G:\.shortcut-targets-by-id\1uyoVEDDA_-tskYM08AbRYIVn64VmCNQm\Bases\02.Volantes\2026\Autorizaciones_202601.txt'
ruta_medicamentos=r'G:\.shortcut-targets-by-id\1e2ieoJan65FyqTpnYzMsthsoUCq0DTBS\6. Resultados en Salud\PYTHON\Proyectos\00.Inversion Salud\03.Fuentes\03.MEDICAMENTOS\MEDICAMENTOS_202601.txt'

In [5]:
class CalculadoraFechas:
    def __init__(self, fecha_base=None):
        self.base = fecha_base or datetime.now().date()
        self.periodo_obj = (self.base - relativedelta(months=1)).replace(day=1)

    def params(self):
        return {
            "anio": self.periodo_obj.strftime("%Y"),
            "mes": self.periodo_obj.strftime("%m"),
            "periodo": self.periodo_obj.strftime("%Y%m")
        }

# --- 2. CLASE FÁBRICA: CONFIGURADOR ---
class FabricaConfig:
    def __init__(self, json_path, params_fecha):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.params_fecha = params_fecha

    def obtener_lectura(self, key):
        info = self.data[key]
        # Inyecta fechas en la plantilla y la base
        ctx = {**info, **self.params_fecha}
        ruta = info['plantilla'].format(**ctx)
        
        return {
            "filepath_or_buffer": ruta,
            "sep": info['delimiter'],
            "encoding": info['encoding'],
            "usecols": info.get('columnas')
        }

# --- 3. CLASE ORQUESTADORA: LOGGING Y CARGA ---
class Orquestador:
    def __init__(self, json_path, fecha_base=None):
        self.fechas = CalculadoraFechas(fecha_base)
        self.fabrica = FabricaConfig(json_path, self.fechas.params())
        self._init_log()

    def _init_log(self):
        os.makedirs('logs', exist_ok=True)
        logging.basicConfig(
            filename=f"logs/carga_{datetime.now().strftime('%Y%m%d')}.log",
            level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.log = logging.getLogger()

    def cargar(self, key):
        try:
            cfg = self.fabrica.obtener_lectura(key)
            if not os.path.exists(cfg['filepath_or_buffer']):
                self.log.warning(f"No existe: {cfg['filepath_or_buffer']}")
                return None
            
            df = pd.read_csv(**cfg)
            self.log.info(f"Éxito: {key} | Filas: {len(df)}")
            return df
        except Exception as e:
            self.log.error(f"Error en {key}: {e}")
            return None

# --- USO ---
if __name__ == "__main__":
    api = Orquestador("scripts\config_rutas.json")
    df_avicena = api.cargar("avicena")

<>:64: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:64: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\mslop\AppData\Local\Temp\ipykernel_20720\3241821076.py:64: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  api = Orquestador("scripts\config_rutas.json")
C:\Users\mslop\AppData\Local\Temp\ipykernel_20720\3241821076.py:55: DtypeWarning: Columns (5,96,107,108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(**cfg)
2026-03-17 08:08:24,449 - INFO - Éxito: avicena | Filas: 1689444


In [6]:
df_avicena.head()

,FOLIO,FECHA_APERTURA_HC,HORA_CIERRE_HC,SUCURSAL,TIPDOC_PACIENTE,NUMDOC_PACIENTE,TIPO_DIAGNOSTICO_1,NOMBRES_MEDICO,APELLIDOS_MEDICO,TIPO_DOC_MEDICO,...,ESPECIALIDAD,CIUDAD,FECHA_ATENCION_DATE,sexo,edad,cod_municip,Usuario_Pac,Usuario_Compartido,SALUD_MENTAL,ESTADO_ENFERMEDAD
0,166689180,18/02/2026,9,SUCURSAL CALLE 100,AS,1080365637,NaN,YOHANNA,SUAREZ BEJARANO,CC,...,ENFERMERIA,BOGOTA D.C.,18/02/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,167348411,27/02/2026,10,CENTRO MEDICO BUCARAMANGA SOTOMAYOR,AS,2031956,NaN,FRANCY TATIANA,VILLABONA GOMEZ,CC,...,MEDICINA GENERAL PROGRAMAS,BUCARAMANGA,27/02/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,166293612,11/02/2026,19,CENTRO MEDICO COLSANITAS EL DORADO,CC,20942026,IMPRESION DIAGNOSTICA,JUAN DIEGO,PACHECO PEDRAZA,CC,...,ENFERMERIA,BOGOTA D.C.,11/02/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,166233179,11/02/2026,9,CENTRO MEDICO COLSANITAS EL DORADO,CC,1000000611,IMPRESION DIAGNOSTICA,LAURA DANIELA,SANTOS GUTIERREZ,CC,...,MEDICINA GENERAL-OPAIN,BOGOTA D.C.,11/02/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,166235083,11/02/2026,10,CENTRO MEDICO COLSANITAS EL DORADO,CC,1000000611,IMPRESION DIAGNOSTICA,JEAN POULL,NEIZA PIRANEQUE,CC,...,AUXILIAR DE ENFERMERIA,BOGOTA D.C.,11/02/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Captura de población

1. primer flujo, realiza un apilado de usuarios finales, realmente es necesario? Vamos a llamar solo usuarios finales el mensual para ver que sucede.
2. El segundo flujo compila el ultimo año de utilizaciones, realmente es necesario todo el año?



In [4]:
# flujo 1, compila los usuarios finales 
# flujo 2, compila las utilizaciones ( se requiere las utilizaciones del ultimo año)
# flujo 3, compila las autorizaciones ( se requiere las autorizaciones del ultimo año)

columnas_interes=['tipo_id','num_id', 'tipo_afil']
usuarios_finales = pd.read_csv(ruta_usuarios_finales + r'\2026_01_Usuarios_EPS_ConsumoInt.txt', sep='|', encoding='latin-1', usecols=columnas_interes)
usuarios_finales.head()

# flujo 4, medicamentos
medicamentos=pd.read_csv(ruta_medicamentos, sep='|', encoding='latin-1')


C:\Users\mslop\AppData\Local\Temp\ipykernel_22268\2350065795.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios_finales = pd.read_csv(ruta_usuarios_finales + r'\2026_01_Usuarios_EPS_ConsumoInt.txt', sep='|', encoding='latin-1', usecols=columnas_interes)
C:\Users\mslop\AppData\Local\Temp\ipykernel_22268\2350065795.py:10: DtypeWarning: Columns (15,16,22,38,61,70,94,103,105,125,136,138,149) have mixed types. Specify dtype option on import or set low_memory=False.
  medicamentos=pd.read_csv(ruta_medicamentos, sep='|', encoding='latin-1')


# Obntención Base RIPS

Se cruza RIPS con los CIE10 de la parámetrica y se asigna el riesgo.
Se conserva los id unicos.

# Consolidación base de salud mental

Se desea traer a todas las personas que tuvieron algún avistamiento de alguno de los CIE-10 que se reportan en el archivo generado para la cohorte. 

In [ ]:
df_avicena_ci=pd.read_csv(r"G:\.shortcut-targets-by-id\1wT-pRaNOECz6KC5hndveeLOIGY381o4T\Alteryx\Proyectos\102.Base_Cacterizacion\04_Salidas\Avicena_Consumo_Int\2025\SALIDA_AVICENA_CONSUMO_INT_"+p_año+p_mes+".txt",delimiter="|",encoding="ISO-8859-1")